<a href="https://colab.research.google.com/github/hillarymaximino/metodos_numericos/blob/main/Atividade_12_M%C3%A9todos_Num%C3%A9ricos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4 — Eliminação Gaussiana com Pivotamento
**Disciplina:** Métodos Numéricos — Engenharia de Computação  
**Instituto Federal da Paraíba (IFPB), Campus Campina Grande**

Este notebook contém os enunciados e as resoluções dos exercícios de fixação, revisão e problemas da Aula 4. Os nomes das variáveis e funções estão em português.

## Bibliotecas utilizadas


In [ ]:
import numpy as np
import scipy.linalg as sla

np.set_printoptions(precision=8, suppress=True)


## Exercício de Fixação 1

**Enunciado:** Explique com suas palavras em que situações a eliminação gaussiana convencional pode gerar problemas computacionais, descrevendo, inclusive, que problemas podem aparecer.

**Resposta:** A eliminação gaussiana convencional pode apresentar problemas quando o pivô escolhido é igual a zero ou tem valor muito pequeno em comparação com os demais elementos da coluna. Se o pivô for zero, ocorre uma divisão por zero e o algoritmo não consegue continuar normalmente. Se for muito pequeno, os multiplicadores podem ficar muito grandes e amplificar erros de arredondamento, causando perda de precisão e resultados pouco confiáveis. O pivotamento parcial reduz esses problemas ao escolher, na coluna atual, o elemento de maior valor absoluto entre a linha do pivô e as linhas abaixo.


## Exercício de Fixação 2

**Enunciado:** Explique o que acontece com o sistema do Exemplo 2 quando \(\varepsilon \to \infty\).

**Resposta:** O sistema é
\[
\begin{bmatrix}\varepsilon&2\\1&2\end{bmatrix}
\begin{bmatrix}x_1\\x_2\end{bmatrix}
=
\begin{bmatrix}4\\3\end{bmatrix}.
\]
Pelas expressões obtidas no material, \(x_2=(3\varepsilon-4)/(2\varepsilon-2)\) e \(x_1=(4-2x_2)/\varepsilon\). Quando \(\varepsilon\to\infty\), temos \(x_2\to 3/2\) e \(x_1\to 0\). Nesse limite, o pivô inicial é muito grande, não pequeno. O problema ilustrado no texto — divisão por pivô muito pequeno e amplificação dos erros — não ocorre nesse limite da mesma forma; a solução tende a \((0,1{,}5)\). O cuidado com o arredondamento continua importante em computação numérica.


## Exercício de Fixação 3

**Enunciado:** Explique por que não podemos simplesmente trocar os elementos relevantes das linhas \(k\) e \(p\) usando o código abaixo. Descreva o que acontece e explique por que a solução apresentada no texto funciona.

```python
for j in range(k, n):
    A[k, j] = A[p, j]
    A[p, j] = A[k, j]
```

O mesmo problema ocorre com a tentativa de usar fatiamento:

```python
A[k, k:] = A[p, k:]
A[p, k:] = A[k, k:]
```

**Resposta:** Na primeira atribuição, os valores da linha `k` são sobrescritos pelos valores da linha `p`. Na segunda, a linha `p` recebe os valores que acabaram de ser copiados para a linha `k`, então as duas linhas ficam iguais e os valores originais da linha `k` são perdidos. A forma correta do NumPy troca as linhas simultaneamente:

```python
A[[k, p]] = A[[p, k]]
```

Assim, o NumPy usa os valores originais das duas linhas para fazer a permuta.


## Exercício de Revisão 4

**Enunciado:** Em que situações é necessário o uso de pivotamento na eliminação gaussiana?

**Resposta:** O pivotamento é necessário quando o pivô atual é zero, pois não é possível dividir por zero, e é recomendável quando o pivô tem valor absoluto muito pequeno em comparação com outros elementos disponíveis na mesma coluna. Nesses casos, trocar linhas pode evitar divisões problemáticas e reduzir a amplificação de erros de arredondamento.


## Exercício de Revisão 5

**Enunciado:** Explique qual a diferença entre a implementação de pivotamento parcial e completo.

**Resposta:** No pivotamento **parcial**, procura-se o maior valor absoluto apenas na coluna atual, considerando a linha do pivô e as linhas abaixo; se necessário, trocam-se linhas. No pivotamento **completo**, procura-se o maior valor absoluto em toda a parte ainda não escalonada da matriz, podendo trocar linhas e colunas. Trocar colunas também muda a ordem das incógnitas, por isso o pivotamento completo exige controlar essa permutação para reconstruir corretamente o vetor solução. O material destaca que o pivotamento parcial é o foco da aula e o completo é menos usado.


## Exercício de Revisão 6

**Enunciado:** Descreva como é feito o processo de eliminação gaussiana com pivotamento parcial.

**Resposta:**
1. Na coluna atual `k`, procura-se, da linha `k` até a última linha, o elemento de maior valor absoluto.
2. Identifica-se a linha `p` onde esse elemento está.
3. Se `p` for diferente de `k`, trocam-se as linhas `p` e `k`.
4. Usa-se o elemento na posição `[k, k]` como pivô para zerar os elementos abaixo dele.
5. Repete-se o processo para as próximas colunas.
6. Ao obter uma matriz triangular superior, resolve-se o sistema por substituição retroativa.


## Problema 7

**Enunciado:** Reescreva o algoritmo de eliminação gaussiana para realizar o pivotamento e implemente uma nova função `escalona` com essa possibilidade, usando `argmax` e `max`.

**Resolução:** A função abaixo trabalha com cópias em ponto flutuante para evitar divisões inteiras e não modificar as entradas originais. Ela retorna a matriz triangular superior e o vetor de termos independentes transformado.


In [ ]:
def escalona_com_pivotamento(matriz, termos_independentes, tolerancia=1e-12):
    A = np.array(matriz, dtype=float, copy=True)
    b = np.array(termos_independentes, dtype=float, copy=True)
    n = len(b)

    if A.shape != (n, n):
        raise ValueError("A matriz deve ser quadrada e compatível com o vetor b.")

    for k in range(n - 1):
        # Escolhe o maior valor absoluto na coluna k, da linha k para baixo.
        indice_relativo = np.abs(A[k:, k]).argmax()
        pivo_linha = indice_relativo + k
        maior_valor = np.abs(A[k:, k]).max()

        if maior_valor < tolerancia:
            raise ValueError("Sistema singular ou pivô numericamente nulo.")

        # Permuta as linhas da matriz e os elementos correspondentes de b.
        if pivo_linha != k:
            A[[k, pivo_linha]] = A[[pivo_linha, k]]
            b[[k, pivo_linha]] = b[[pivo_linha, k]]

        for linha in range(k + 1, n):
            multiplicador = A[linha, k] / A[k, k]
            A[linha, k:] = A[linha, k:] - multiplicador * A[k, k:]
            b[linha] = b[linha] - multiplicador * b[k]
            A[linha, k] = 0.0

    if abs(A[n - 1, n - 1]) < tolerancia:
        raise ValueError("Sistema singular ou último pivô numericamente nulo.")

    return A, b


def substituicao_retroativa(matriz_triangular, termos_transformados):
    A = np.asarray(matriz_triangular, dtype=float)
    b = np.asarray(termos_transformados, dtype=float)
    n = len(b)
    solucao = np.zeros(n)

    for i in range(n - 1, -1, -1):
        soma = np.dot(A[i, i + 1:], solucao[i + 1:])
        if abs(A[i, i]) < 1e-12:
            raise ValueError("Pivô nulo durante a substituição retroativa.")
        solucao[i] = (b[i] - soma) / A[i, i]

    return solucao


# Teste simples com uma matriz que precisa de troca de linhas.
matriz_teste = np.array([[1, 1, 1], [4, 4, 2], [2, 1, -1]], dtype=float)
vetor_teste = np.array([1, 2, 0], dtype=float)

matriz_escalonada, vetor_escalonado = escalona_com_pivotamento(matriz_teste, vetor_teste)
solucao_teste = substituicao_retroativa(matriz_escalonada, vetor_escalonado)

print("Matriz escalonada:")
print(matriz_escalonada)
print("Vetor escalonado:", vetor_escalonado)
print("Solução:", solucao_teste)
print("Conferência A @ x:", matriz_teste @ solucao_teste)


## Problema 8

**Enunciado:** Adapte a implementação do problema anterior para utilizar a função `numpy.roll()` na operação de permutação das linhas, nos casos necessários.

**Resolução:** `np.roll(vetor, 1)` desloca os elementos uma posição para a direita, levando o último elemento para o início. Para trocar as linhas `k` e `p`, uma forma simples é montar a ordem desejada das linhas e aplicá-la à matriz e ao vetor `b`. A função abaixo usa `np.roll` para rotacionar temporariamente o trecho de linhas e, depois, coloca a linha escolhida na posição do pivô.


In [ ]:
def escalona_com_roll(matriz, termos_independentes, tolerancia=1e-12):
    A = np.array(matriz, dtype=float, copy=True)
    b = np.array(termos_independentes, dtype=float, copy=True)
    n = len(b)

    if A.shape != (n, n):
        raise ValueError("A matriz deve ser quadrada e compatível com o vetor b.")

    for k in range(n - 1):
        pivo_linha = np.abs(A[k:, k]).argmax() + k

        if abs(A[pivo_linha, k]) < tolerancia:
            raise ValueError("Sistema singular ou pivô numericamente nulo.")

        if pivo_linha != k:
            # Rotaciona o trecho de linhas k até pivo_linha e faz o mesmo em b.
            A[k:pivo_linha + 1] = np.roll(A[k:pivo_linha + 1], 1, axis=0)
            b[k:pivo_linha + 1] = np.roll(b[k:pivo_linha + 1], 1)

        for linha in range(k + 1, n):
            multiplicador = A[linha, k] / A[k, k]
            A[linha, k:] -= multiplicador * A[k, k:]
            b[linha] -= multiplicador * b[k]
            A[linha, k] = 0.0

    if abs(A[-1, -1]) < tolerancia:
        raise ValueError("Sistema singular ou último pivô numericamente nulo.")

    return A, b


matriz_roll, vetor_roll = escalona_com_roll(matriz_teste, vetor_teste)
solucao_roll = substituicao_retroativa(matriz_roll, vetor_roll)
print("Matriz escalonada com roll:")
print(matriz_roll)
print("Vetor escalonado:", vetor_roll)
print("Solução:", solucao_roll)
print("Conferência A @ x:", matriz_teste @ solucao_roll)


## Problema 9

**Enunciado:** Resolva o sistema \(A x=b\), com
\[
A=\begin{bmatrix}4&2&1\\9&3&1\\25&5&1\end{bmatrix},
\qquad
b=\begin{bmatrix}0.693147\\1.098612\\1.609438\end{bmatrix},
\]
usando primeiro eliminação gaussiana convencional e depois eliminação gaussiana com pivotamento parcial. No cálculo manual, todos os valores intermediários devem ser arredondados para **quatro algarismos significativos**, sem frações. Para comparar as soluções, calcule o resíduo \(r=b-Ax\) e a norma \(\|r\|_\infty\) com alta precisão, sem arredondamento.

**Resolução:** O código abaixo simula o arredondamento para quatro algarismos significativos após cada operação aritmética usada no escalonamento e na substituição retroativa. Depois, calcula os resíduos usando os valores originais de `A` e `b`, sem arredondar esses cálculos.


In [ ]:
def arredondar_significativos(valor, algarismos=4):
    valor = float(valor)
    if valor == 0 or not np.isfinite(valor):
        return valor
    casas = algarismos - 1 - int(np.floor(np.log10(abs(valor))))
    return float(np.round(valor, casas))


def operar_arredondando(a, b, operacao):
    if operacao == "soma":
        return arredondar_significativos(a + b)
    if operacao == "subtracao":
        return arredondar_significativos(a - b)
    if operacao == "multiplicacao":
        return arredondar_significativos(a * b)
    if operacao == "divisao":
        return arredondar_significativos(a / b)
    raise ValueError("Operação desconhecida.")


def eliminacao_com_arredondamento(matriz, termos, usar_pivotamento=False):
    A = np.array(matriz, dtype=float, copy=True)
    b = np.array(termos, dtype=float, copy=True)
    n = len(b)

    A = np.vectorize(arredondar_significativos)(A)
    b = np.vectorize(arredondar_significativos)(b)

    for k in range(n - 1):
        if usar_pivotamento:
            p = np.abs(A[k:, k]).argmax() + k
            if p != k:
                A[[k, p]] = A[[p, k]]
                b[[k, p]] = b[[p, k]]

        if A[k, k] == 0:
            raise ZeroDivisionError("Pivô zero na eliminação convencional.")

        for i in range(k + 1, n):
            multiplicador = operar_arredondando(A[i, k], A[k, k], "divisao")
            for j in range(k, n):
                produto = operar_arredondando(multiplicador, A[k, j], "multiplicacao")
                A[i, j] = operar_arredondando(A[i, j], produto, "subtracao")
            produto_b = operar_arredondando(multiplicador, b[k], "multiplicacao")
            b[i] = operar_arredondando(b[i], produto_b, "subtracao")
            A[i, k] = 0.0

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma = 0.0
        for j in range(i + 1, n):
            produto = operar_arredondando(A[i, j], x[j], "multiplicacao")
            soma = operar_arredondando(soma, produto, "soma")
        numerador = operar_arredondando(b[i], soma, "subtracao")
        x[i] = operar_arredondando(numerador, A[i, i], "divisao")

    return A, b, x


matriz = np.array([[4, 2, 1],
                   [9, 3, 1],
                   [25, 5, 1]], dtype=float)
termos = np.array([0.693147, 1.098612, 1.609438], dtype=float)

# Caso 1: eliminação convencional
try:
    matriz_convencional, termos_convencionais, solucao_convencional = eliminacao_com_arredondamento(
        matriz, termos, usar_pivotamento=False
    )
    residuo_convencional = termos - matriz @ solucao_convencional
    norma_convencional = np.linalg.norm(residuo_convencional, ord=np.inf)

    print("=== Eliminação convencional ===")
    print("Matriz escalonada:")
    print(matriz_convencional)
    print("Termos transformados:", termos_convencionais)
    print("Solução aproximada:", solucao_convencional)
    print("Resíduo calculado sem arredondamento:", residuo_convencional)
    print("Norma infinito do resíduo:", format(norma_convencional, ".16e"))
except (ZeroDivisionError, ValueError) as erro:
    print("Falha na eliminação convencional:", erro)

# Caso 2: eliminação com pivotamento parcial
matriz_pivotada, termos_pivotados, solucao_pivotada = eliminacao_com_arredondamento(
    matriz, termos, usar_pivotamento=True
)
residuo_pivotado = termos - matriz @ solucao_pivotada
norma_pivotada = np.linalg.norm(residuo_pivotado, ord=np.inf)

print("\n=== Eliminação com pivotamento parcial ===")
print("Matriz escalonada:")
print(matriz_pivotada)
print("Termos transformados:", termos_pivotados)
print("Solução aproximada:", solucao_pivotada)
print("Resíduo calculado sem arredondamento:", residuo_pivotado)
print("Norma infinito do resíduo:", format(norma_pivotada, ".16e"))

# Referência adicional: solução de alta precisão do sistema original, sem arredondamento intermediário.
solucao_referencia = np.linalg.solve(matriz, termos)
print("\nSolução de referência (NumPy, sem arredondamento manual):", solucao_referencia)


## Observação final

- Os enunciados foram transcritos e organizados a partir do PDF da Aula 4.
- As respostas conceituais estão escritas em linguagem direta.
- Nos Problemas 7 e 8, as funções retornam a matriz triangular superior e o vetor transformado; a função `substituicao_retroativa` calcula a solução.
- No Problema 9, o arredondamento para quatro algarismos significativos é simulado durante as operações. Os resíduos são calculados depois com os dados originais e sem arredondamento.
